In [4]:
import tushare as ts
import pandas as pd
import time
import os

print('所有库导入成功 ✓')

所有库导入成功 ✓


In [5]:
# 把引号里换成你重置后的新 token
MY_TOKEN = '79ef0fa884254e282de0a15ae4f9647f722e6f28d61522def56de635'

ts.set_token(MY_TOKEN)
pro = ts.pro_api()

print('Tushare 连接成功 ✓')


Tushare 连接成功 ✓


In [6]:
import os

# 直接指定你的项目文件夹路径，不依赖工作目录
os.chdir('/Users/liujingxin/Documents/Quant_Project')

# 确认切换成功
print(f'当前工作目录：{os.getcwd()}')

# 创建 data 文件夹
os.makedirs('data', exist_ok=True)
print('data/ 文件夹已准备好 ✓')

当前工作目录：/Users/liujingxin/Documents/Quant_Project
data/ 文件夹已准备好 ✓


In [7]:
# stock_basic 接口返回所有上市公司基本信息
# list_status='L' 只要上市中的股票，排除已退市(D)和暂停上市(P)
stock_basic = pro.stock_basic(
    exchange='',
    list_status='L',
    fields='ts_code, name, industry, list_date'
)

# 单独保存行业分类，后面做行业中性化时会用到
stock_basic.to_csv('data/stock_industry.csv', index=False)

print(f'共获取到 {len(stock_basic)} 只上市股票')
stock_basic.head()

共获取到 5525 只上市股票


,ts_code,name,industry,list_date
0,000001.SZ,平安银行,银行,19910403
1,000002.SZ,万科Ａ,全国地产,19910129
2,000004.SZ,*ST国华,软件服务,19901201
3,000006.SZ,深振业Ａ,区域地产,19920427
4,000007.SZ,全新好,其他商业,19920413


In [8]:
# 获取2020-2025年所有开市日期
trade_cal = pro.trade_cal(
    exchange='SSE',
    start_date='20200101',
    end_date='20251231',
    is_open='1',         # 只要开市的日期
    fields='cal_date'
)

# 把日期字符串转成 pandas 能识别的日期格式
# 转换后才能做"按年月分组"这类操作
trade_cal['cal_date'] = pd.to_datetime(trade_cal['cal_date'])

# 按年月分组，每组取最后一天 = 月末最后一个交易日
# to_period('M') 把日期转成"年月"格式，比如 2020-01
month_end_dates = (
    trade_cal
    .groupby(trade_cal['cal_date'].dt.to_period('M'))
    ['cal_date']
    .max()
    .reset_index(drop=True)
)

# 转回 Tushare 要求的字符串格式 'YYYYMMDD'
month_end_str = month_end_dates.dt.strftime('%Y%m%d').tolist()

print(f'共 {len(month_end_str)} 个月末交易日')
print(f'第一个：{month_end_str[0]}，最后一个：{month_end_str[-1]}')

共 72 个月末交易日
第一个：20200123，最后一个：20251231


In [9]:
# 用来存每个月数据的列表
# 每次循环往里加一张表，最后再合并成一张大表
all_monthly_data = []

# 要拉的字段
fields_to_fetch = 'ts_code,trade_date,close,pe_ttm,pb,total_mv,circ_mv'

# enumerate() 同时给我们"序号 i"和"日期 date"
# 这样可以打印进度，知道跑到第几个月
for i, date in enumerate(month_end_str):

    print(f'正在拉取第 {i+1}/{len(month_end_str)} 个月：{date}', end='\r')

    # try...except：如果某个月拉取失败，不崩溃，跳过继续跑
    try:
        df = pro.daily_basic(
            trade_date=date,
            fields=fields_to_fetch
        )

        # 只有数据不为空才加入列表
        if len(df) > 0:
            all_monthly_data.append(df)

    except Exception as e:
        print(f'\n{date} 拉取失败：{e}')

    # 每次调用后等0.5秒，避免触发 API 限速
    time.sleep(0.5)

print(f'\n完成！成功获取 {len(all_monthly_data)} 个月的数据 ✓')

正在拉取第 72/72 个月：20251231
完成！成功获取 72 个月的数据 ✓


In [10]:
# pd.concat() 把列表里所有表纵向拼成一张大表
raw_data = pd.concat(all_monthly_data, ignore_index=True)

# 日期列转成日期格式
raw_data['trade_date'] = pd.to_datetime(raw_data['trade_date'])

# 按股票代码 + 日期排序，数据更整齐
raw_data = raw_data.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)

# 保存成 CSV
raw_data.to_csv('data/raw_monthly_data.csv', index=False)

print(f'已保存到 data/raw_monthly_data.csv')
print(f'总行数：{len(raw_data):,}')
print(f'时间范围：{raw_data["trade_date"].min()} 到 {raw_data["trade_date"].max()}')
print(f'股票数量：{raw_data["ts_code"].nunique():,} 只')

raw_data.head(10)

已保存到 data/raw_monthly_data.csv
总行数：349,215
时间范围：2020-01-23 00:00:00 到 2025-12-31 00:00:00
股票数量：5,675 只


,ts_code,trade_date,close,pe_ttm,pb,total_mv,circ_mv
0,000001.SZ,2020-01-23,15.54,10.6958,1.1045,3.015680e+07,3.015654e+07
1,000001.SZ,2020-02-28,14.50,9.9800,1.0306,2.813858e+07,2.813834e+07
2,000001.SZ,2020-03-31,12.80,8.8099,0.9098,2.483958e+07,2.483936e+07
3,000001.SZ,2020-04-30,13.93,9.2270,0.9572,2.703244e+07,2.703221e+07
4,000001.SZ,2020-05-29,13.00,8.6110,0.7687,2.522769e+07,2.522748e+07
5,000001.SZ,2020-06-30,12.80,8.4785,0.8929,2.483958e+07,2.483936e+07
6,000001.SZ,2020-07-31,13.34,8.8362,0.9306,2.588749e+07,2.588727e+07
7,000001.SZ,2020-08-31,15.08,11.0556,1.0398,2.926412e+07,2.926388e+07
8,000001.SZ,2020-09-30,15.17,11.1216,1.0460,2.943878e+07,2.943853e+07
9,000001.SZ,2020-10-30,17.75,12.7708,1.1926,3.444550e+07,3.444521e+07


In [11]:
# 读取刚才保存的行业分类
stock_industry = pd.read_csv('data/stock_industry.csv')
stock_industry = stock_industry[['ts_code', 'industry']]

# merge 就像 Excel 的 VLOOKUP
# 用 ts_code 作为 key，把行业信息匹配进来
# how='left' → 保留 raw_data 所有行，找不到行业就填 NaN
raw_data = raw_data.merge(stock_industry, on='ts_code', how='left')

# 覆盖保存
raw_data.to_csv('data/raw_monthly_data.csv', index=False)

print(f'行业分类合并完成 ✓')
print(f'共 {raw_data["industry"].nunique()} 个行业')

行业分类合并完成 ✓
共 110 个行业


In [12]:
print('各列缺失值情况：')
missing = pd.DataFrame({
    '缺失数量': raw_data.isnull().sum(),
    '缺失比例%': (raw_data.isnull().sum() / len(raw_data) * 100).round(2)
})
print(missing)

print()
print('PE 和 PB 基本统计：')
print(raw_data[['pe_ttm', 'pb', 'total_mv']].describe())

各列缺失值情况：
             缺失数量  缺失比例%
ts_code         0   0.00
trade_date      0   0.00
close           0   0.00
pe_ttm      73996  21.19
pb           2535   0.73
total_mv        0   0.00
circ_mv         0   0.00
industry     8569   2.45

PE 和 PB 基本统计：
              pe_ttm             pb      total_mv
count  275219.000000  346680.000000  3.492150e+05
mean      122.415367       4.213365  1.871409e+06
std      2298.306101      30.785462  8.147465e+06
min         0.176800       0.050300  5.761991e+03
25%        20.156200       1.611675  3.066764e+05
50%        34.551200       2.503600  5.389077e+05
75%        64.787300       4.011600  1.224967e+06
max    539273.338300    9234.002200  2.890455e+08
